# NYC Taxi & Limousine Commission — ETL Pipeline

A production-style pipeline that ingests raw Parquet data from the TLC, applies **physics-based cleaning** with NumPy, performs **financial auditing** with Pandas, engineers mobility features, and delivers a Gold dataset plus KPI summary tables.

---
| Stage | Tool | Purpose |
|---|---|---|
| Extract | `urllib` + `pyarrow` | Download & load raw Parquet |
| Physics Filter | `NumPy` | Remove impossible trips |
| Financial Audit | `Pandas` | Clean fares, compute tip % |
| Feature Engineering | `Pandas` + `NumPy` | Temporal features, borough mapping |
| Load | `pyarrow` | Write Gold Parquet + KPI CSVs |
| Analyse | `matplotlib` + `seaborn` | KPI visualisations |

## Section 0 — Setup & Configuration

In [ ]:
import sys
import subprocess

# Install dependencies if missing
required = ['pandas', 'numpy', 'pyarrow', 'matplotlib', 'seaborn', 'requests']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + required)

In [ ]:
import os
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 40)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (10, 5)})

# ── Constants ─────────────────────────────────────────────────────────────────
JFK_ZONE_ID        = 132
LGA_ZONE_ID        = 138    # LaGuardia
AIRPORT_SURCHARGE  = 1.75   # USD — standard NYC TLC airport improvement fee
MAX_DISTANCE_MI    = 100    # miles — beyond this is physically unreasonable in NYC metro
MAX_SPEED_MPH      = 80     # mph  — NYC highways cap at 55 mph; 80 is a generous ceiling
RUSH_AM_START      = 7
RUSH_AM_END        = 9
RUSH_PM_START      = 16
RUSH_PM_END        = 19

# ── Payment type mapping (TLC encoding) ───────────────────────────────────────
PAYMENT_MAP = {
    1: 'Credit Card',
    2: 'Cash',
    3: 'No Charge',
    4: 'Dispute',
    5: 'Unknown',
    6: 'Voided Trip',
}

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR       = os.getcwd()
RAW_DIR        = os.path.join(BASE_DIR, 'data', 'raw')
LOOKUP_DIR     = os.path.join(BASE_DIR, 'data', 'lookup')
GOLD_DIR       = os.path.join(BASE_DIR, 'output', 'gold')
KPI_DIR        = os.path.join(BASE_DIR, 'output', 'kpi')

PARQUET_URL    = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet'
ZONE_URL       = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

PARQUET_PATH   = os.path.join(RAW_DIR,    'yellow_tripdata_2024-01.parquet')
ZONE_PATH      = os.path.join(LOOKUP_DIR, 'taxi_zone_lookup.csv')
GOLD_PATH      = os.path.join(GOLD_DIR,   'nyc_taxi_gold.parquet')

for d in [RAW_DIR, LOOKUP_DIR, GOLD_DIR, KPI_DIR]:
    os.makedirs(d, exist_ok=True)

print('Configuration loaded.')
print(f'  Base directory : {BASE_DIR}')
print(f'  JFK Zone ID    : {JFK_ZONE_ID}')
print(f'  LaGuardia ID   : {LGA_ZONE_ID}')
print(f'  Max distance   : {MAX_DISTANCE_MI} miles')
print(f'  Max speed      : {MAX_SPEED_MPH} mph')

## Section 1 — Extraction

Downloads the Yellow Taxi Parquet file (~50 MB compressed) and the Taxi Zone Lookup CSV from the official TLC CloudFront CDN.

In [ ]:
def download_if_missing(url: str, dest: str) -> None:
    """Download url → dest only when the file is not already present."""
    if os.path.exists(dest):
        size_mb = os.path.getsize(dest) / 1_048_576
        print(f'  [SKIP]  {os.path.basename(dest):45s}  ({size_mb:.1f} MB already on disk)')
        return
    print(f'  [DOWN]  {os.path.basename(dest):45s}  downloading …', end='', flush=True)
    urllib.request.urlretrieve(url, dest)
    size_mb = os.path.getsize(dest) / 1_048_576
    print(f'  done  ({size_mb:.1f} MB)')


print('Fetching data sources:')
download_if_missing(PARQUET_URL, PARQUET_PATH)
download_if_missing(ZONE_URL,    ZONE_PATH)
print('\nAll sources ready.')

In [ ]:
print('Loading Parquet …')
df_raw = pd.read_parquet(PARQUET_PATH, engine='pyarrow')
zone_lookup = pd.read_csv(ZONE_PATH)

print(f'\nRaw trip records  : {df_raw.shape[0]:,} rows  ×  {df_raw.shape[1]} columns')
print(f'Zone lookup table : {zone_lookup.shape[0]} rows  ×  {zone_lookup.shape[1]} columns')

print('\n── Column dtypes ──')
print(df_raw.dtypes.to_string())

print('\n── Null counts (top 10) ──')
null_counts = df_raw.isnull().sum().sort_values(ascending=False)
print(null_counts[null_counts > 0].to_string())

df_raw.head(3)

In [ ]:
print('── Zone lookup sample ──')
print(zone_lookup.head(10).to_string(index=False))

## Section 2 — Physics Filter (NumPy)

We build three **vectorized boolean masks** and combine them with `np.logical_and.reduce` for a single-pass filter — no Python-level loops.

| Guard | Rule | Rationale |
|---|---|---|
| Distance | `0 < trip_distance ≤ 100 mi` | Zero or negative distance is impossible; >100 mi in NYC metro is unreasonable |
| Speed | `avg_speed ≤ 80 mph` | NYC highway limit is 55 mph; 80 mph is a generous sensor-error ceiling |
| Time logic | `dropoff > pickup` | Negative or zero duration violates causality |

In [ ]:
df = df_raw.copy()
RAW_COUNT = len(df)

# ── Pre-compute duration in hours (needed for speed) ─────────────────────────
df['duration_hrs'] = (
    df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
).dt.total_seconds() / 3600

# ── Mask 1: Distance guard ────────────────────────────────────────────────────
mask_distance = np.logical_and(
    df['trip_distance'].to_numpy() > 0,
    df['trip_distance'].to_numpy() <= MAX_DISTANCE_MI
)

# ── Mask 2: Speed guard ───────────────────────────────────────────────────────
#   Avoid division by zero: treat duration <= 0 as invalid (handled by mask 3)
safe_duration = np.where(df['duration_hrs'].to_numpy() > 0,
                         df['duration_hrs'].to_numpy(), np.nan)
avg_speed = df['trip_distance'].to_numpy() / safe_duration
mask_speed = avg_speed <= MAX_SPEED_MPH  # NaN comparisons evaluate to False → filtered out

# ── Mask 3: Time-logic guard ──────────────────────────────────────────────────
mask_time = df['duration_hrs'].to_numpy() > 0

# ── Combine all masks (logical AND) ──────────────────────────────────────────
combined_mask = np.logical_and.reduce([mask_distance, mask_speed, mask_time])
df = df.loc[combined_mask].copy()

AFTER_PHYSICS = len(df)
physics_removed = RAW_COUNT - AFTER_PHYSICS

print('── Physics Filter Audit ─────────────────────────────────────────────')
print(f'  Raw rows             : {RAW_COUNT:>10,}')
print(f'  Removed (distance)   : {(~mask_distance).sum():>10,}  ({(~mask_distance).mean()*100:.2f}%)')
print(f'  Removed (speed)      : {(~mask_speed).sum():>10,}  ({(~mask_speed).mean()*100:.2f}%)')
print(f'  Removed (time logic) : {(~mask_time).sum():>10,}  ({(~mask_time).mean()*100:.2f}%)')
print(f'  Removed (combined)   : {physics_removed:>10,}  ({physics_removed/RAW_COUNT*100:.2f}%)')
print(f'  Remaining rows       : {AFTER_PHYSICS:>10,}')

## Section 3 — Financial Audit (Pandas)

In [ ]:
AFTER_PHYSICS_COUNT = len(df)

# ── 3a. Remove negative fares ─────────────────────────────────────────────────
neg_fare_mask = (df['fare_amount'] >= 0) & (df['total_amount'] >= 0)
n_neg_fare = (~neg_fare_mask).sum()
df = df.loc[neg_fare_mask].copy()

print(f'Negative-fare rows removed : {n_neg_fare:,}')
print(f'Rows after fare filter     : {len(df):,}')

In [ ]:
# ── 3b. Tip percentage ────────────────────────────────────────────────────────
#   Use np.where to safely handle zero-fare rows (avoid NaN propagation)
df['tip_pct'] = np.where(
    df['fare_amount'] > 0,
    (df['tip_amount'] / df['fare_amount']) * 100,
    0.0
)

print('── Tip Percentage Summary ───────────────────────────────────────────')
print(df['tip_pct'].describe().round(2).to_string())
print(f"\nTrips with no tip    : {(df['tip_pct'] == 0).sum():,}  ({(df['tip_pct'] == 0).mean()*100:.1f}%)")
print(f"Trips tipping >20%%  : {(df['tip_pct'] > 20).sum():,}  ({(df['tip_pct'] > 20).mean()*100:.1f}%)")

In [ ]:
# ── 3c. Airport surcharge validation ─────────────────────────────────────────
#   TLC mandates a $1.75 airport improvement fee for trips to/from JFK & LGA.
#   We validate using np.where: if destination is an airport zone, fee must be >= 1.75.
airport_zones = np.array([JFK_ZONE_ID, LGA_ZONE_ID])
is_airport_dest = np.isin(df['DOLocationID'].to_numpy(), airport_zones)

# airport_fee column name in TLC data (present from 2022 onward)
fee_col = 'Airport_fee' if 'Airport_fee' in df.columns else 'airport_fee'

if fee_col in df.columns:
    df['airport_fee_valid'] = np.where(
        is_airport_dest,
        df[fee_col].to_numpy() >= AIRPORT_SURCHARGE,
        True   # non-airport trips always pass
    )
    total_airport = is_airport_dest.sum()
    discrepant     = (~df.loc[is_airport_dest, 'airport_fee_valid']).sum()
    print('── Airport Surcharge Audit ──────────────────────────────────────────')
    print(f'  Trips destined for JFK/LGA : {total_airport:>10,}')
    print(f'  Fee discrepancies found    : {discrepant:>10,}  ({discrepant/total_airport*100:.2f}% of airport trips)')
else:
    df['airport_fee_valid'] = True
    print(f'Column "{fee_col}" not present in this dataset — skipping surcharge audit.')

AFTER_FINANCIAL = len(df)
print(f'\nRows after financial audit : {AFTER_FINANCIAL:,}')

## Section 4 — Feature Engineering

In [ ]:
# ── 4a. Temporal features ─────────────────────────────────────────────────────
df['hour_of_day'] = df['tpep_pickup_datetime'].dt.hour
df['day_of_week'] = df['tpep_pickup_datetime'].dt.day_name()
df['month']       = df['tpep_pickup_datetime'].dt.month
df['date']        = df['tpep_pickup_datetime'].dt.date

# ── 4b. Rush hour flag (vectorized with np.where) ────────────────────────────
hour_arr = df['hour_of_day'].to_numpy()
df['is_rush_hour'] = np.where(
    ((hour_arr >= RUSH_AM_START) & (hour_arr <= RUSH_AM_END)) |
    ((hour_arr >= RUSH_PM_START) & (hour_arr <= RUSH_PM_END)),
    True,
    False
)

print('── Temporal Feature Sample ──────────────────────────────────────────')
print(df[['tpep_pickup_datetime', 'hour_of_day', 'day_of_week', 'is_rush_hour']].head(8).to_string(index=False))
print(f"\nRush-hour trips : {df['is_rush_hour'].sum():,}  ({df['is_rush_hour'].mean()*100:.1f}% of cleaned trips)")

In [ ]:
# ── 4c. Borough mapping via pd.merge ─────────────────────────────────────────
#   Merge twice: once for pickup zone, once for dropoff zone.

zone_pu = zone_lookup[['LocationID', 'Borough', 'Zone']].rename(columns={
    'LocationID': 'PULocationID',
    'Borough':    'pickup_borough',
    'Zone':       'pickup_zone',
})

zone_do = zone_lookup[['LocationID', 'Borough', 'Zone']].rename(columns={
    'LocationID': 'DOLocationID',
    'Borough':    'dropoff_borough',
    'Zone':       'dropoff_zone',
})

df = (
    df
    .merge(zone_pu, on='PULocationID', how='left')
    .merge(zone_do, on='DOLocationID', how='left')
)

print('── Borough Distribution (Pickup) ────────────────────────────────────')
print(df['pickup_borough'].value_counts().to_string())
print('\n── Borough Distribution (Dropoff) ───────────────────────────────────')
print(df['dropoff_borough'].value_counts().to_string())

In [ ]:
# ── 4d. Revenue per mile (efficiency metric) ──────────────────────────────────
df['revenue_per_mile'] = np.where(
    df['trip_distance'] > 0,
    df['total_amount'] / df['trip_distance'],
    np.nan
)

# ── 4e. Payment type label ────────────────────────────────────────────────────
df['payment_label'] = df['payment_type'].map(PAYMENT_MAP).fillna('Unknown')

print('── Revenue per Mile Summary ─────────────────────────────────────────')
print(df['revenue_per_mile'].describe().round(2).to_string())
print(f"\nAverage revenue/mile : ${df['revenue_per_mile'].mean():.2f}")

## Section 5 — Gold Dataset Output

Write the refined dataset as a Parquet file and print the full filter-stage audit log.

In [ ]:
df_gold = df.copy()
df_gold.to_parquet(GOLD_PATH, index=False, engine='pyarrow')
gold_size_mb = os.path.getsize(GOLD_PATH) / 1_048_576

print('════════════════════════════════════════════════════════════════════')
print('                    FILTER STAGE AUDIT LOG                          ')
print('════════════════════════════════════════════════════════════════════')
print(f'  Stage 0 — Raw ingestion        : {RAW_COUNT:>10,} rows  (100.00%)')

after_physics_n = AFTER_PHYSICS
print(f'  Stage 1 — Physics filter       : {after_physics_n:>10,} rows  '
      f'({after_physics_n/RAW_COUNT*100:.2f}%)  '
      f'[-{RAW_COUNT - after_physics_n:,} removed]')

print(f'  Stage 2 — Financial audit      : {AFTER_FINANCIAL:>10,} rows  '
      f'({AFTER_FINANCIAL/RAW_COUNT*100:.2f}%)  '
      f'[-{after_physics_n - AFTER_FINANCIAL:,} removed]')

final_n = len(df_gold)
print(f'  Stage 3 — Gold dataset (final) : {final_n:>10,} rows  '
      f'({final_n/RAW_COUNT*100:.2f}%)')
print('════════════════════════════════════════════════════════════════════')
print(f'  Gold file written to : {GOLD_PATH}')
print(f'  Gold file size       : {gold_size_mb:.1f} MB')
print('════════════════════════════════════════════════════════════════════')

## Section 6 — KPI Summary Tables & Visualisations

Four analytical deliverables answering the core business questions.

In [ ]:
# Helper to save KPI tables
def save_kpi(df_kpi: pd.DataFrame, name: str) -> None:
    path = os.path.join(KPI_DIR, f'{name}.csv')
    df_kpi.to_csv(path)
    print(f'  Saved: {path}')

### KPI 1 — Busiest Hour (Demand Peaks)

In [ ]:
kpi_busiest_hour = (
    df_gold
    .groupby('hour_of_day')
    .size()
    .reset_index(name='trip_count')
    .sort_values('hour_of_day')
)

save_kpi(kpi_busiest_hour.set_index('hour_of_day'), 'kpi_busiest_hour')

fig, ax = plt.subplots(figsize=(12, 5))

colors = [
    '#e74c3c' if (
        (RUSH_AM_START <= h <= RUSH_AM_END) or (RUSH_PM_START <= h <= RUSH_PM_END)
    ) else '#3498db'
    for h in kpi_busiest_hour['hour_of_day']
]

bars = ax.bar(kpi_busiest_hour['hour_of_day'], kpi_busiest_hour['trip_count'],
              color=colors, edgecolor='white', linewidth=0.5)

ax.set_xlabel('Hour of Day (24 h)', fontsize=12)
ax.set_ylabel('Number of Trips', fontsize=12)
ax.set_title('NYC Yellow Taxi — Busiest Hours (Jan 2024)', fontsize=14, fontweight='bold')
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='Rush Hour'),
                   Patch(facecolor='#3498db', label='Off-Peak')]
ax.legend(handles=legend_elements, loc='upper left')

peak_hour = kpi_busiest_hour.loc[kpi_busiest_hour['trip_count'].idxmax(), 'hour_of_day']
peak_count = kpi_busiest_hour['trip_count'].max()
ax.annotate(f'Peak: {peak_hour:02d}:00\n({peak_count/1000:.1f}k trips)',
            xy=(peak_hour, peak_count),
            xytext=(peak_hour + 1.5, peak_count * 0.95),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(KPI_DIR, 'kpi_busiest_hour.png'), bbox_inches='tight')
plt.show()

print('\n── Busiest Hour Summary ─────────────────────────────────────────────')
print(kpi_busiest_hour.sort_values('trip_count', ascending=False).head(5).to_string(index=False))

### KPI 2 — Revenue by Borough

In [ ]:
kpi_borough_revenue = (
    df_gold
    .groupby('pickup_borough')['total_amount']
    .agg(total_revenue='sum', avg_revenue='mean', trip_count='count')
    .reset_index()
    .sort_values('total_revenue', ascending=True)
)

save_kpi(kpi_borough_revenue.set_index('pickup_borough'), 'kpi_revenue_by_borough')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total revenue
palette = sns.color_palette('Blues_d', len(kpi_borough_revenue))
bars = axes[0].barh(kpi_borough_revenue['pickup_borough'],
                     kpi_borough_revenue['total_revenue'] / 1_000_000,
                     color=palette)
axes[0].set_xlabel('Total Revenue (USD millions)', fontsize=11)
axes[0].set_title('Total Revenue by Pickup Borough', fontsize=13, fontweight='bold')
for bar, val in zip(bars, kpi_borough_revenue['total_revenue'] / 1_000_000):
    axes[0].text(val + 0.05, bar.get_y() + bar.get_height()/2,
                 f'${val:.1f}M', va='center', fontsize=9)

# Average fare
kpi_sorted_avg = kpi_borough_revenue.sort_values('avg_revenue', ascending=True)
palette2 = sns.color_palette('Greens_d', len(kpi_sorted_avg))
bars2 = axes[1].barh(kpi_sorted_avg['pickup_borough'],
                      kpi_sorted_avg['avg_revenue'],
                      color=palette2)
axes[1].set_xlabel('Average Revenue per Trip (USD)', fontsize=11)
axes[1].set_title('Avg Revenue per Trip by Borough', fontsize=13, fontweight='bold')
for bar, val in zip(bars2, kpi_sorted_avg['avg_revenue']):
    axes[1].text(val + 0.2, bar.get_y() + bar.get_height()/2,
                 f'${val:.2f}', va='center', fontsize=9)

plt.suptitle('NYC Taxi — Driver Profitability by Borough (Jan 2024)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(KPI_DIR, 'kpi_revenue_by_borough.png'), bbox_inches='tight')
plt.show()

print('\n── Revenue by Borough ───────────────────────────────────────────────')
print(kpi_borough_revenue.sort_values('total_revenue', ascending=False).to_string(index=False))

### KPI 3 — Efficiency Index (Revenue per Mile by Hour)

In [ ]:
kpi_efficiency = (
    df_gold
    .groupby('hour_of_day')['revenue_per_mile']
    .agg(avg_rev_per_mile='mean', median_rev_per_mile='median', trip_count='count')
    .reset_index()
    .sort_values('hour_of_day')
)

save_kpi(kpi_efficiency.set_index('hour_of_day'), 'kpi_efficiency_index')

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(kpi_efficiency['hour_of_day'], kpi_efficiency['avg_rev_per_mile'],
        color='#2ecc71', linewidth=2.5, marker='o', markersize=5, label='Mean $/mile')
ax.plot(kpi_efficiency['hour_of_day'], kpi_efficiency['median_rev_per_mile'],
        color='#e67e22', linewidth=2, linestyle='--', marker='s', markersize=4,
        label='Median $/mile')

# Shade rush hours
ax.axvspan(RUSH_AM_START, RUSH_AM_END, alpha=0.12, color='red', label='Rush hours')
ax.axvspan(RUSH_PM_START, RUSH_PM_END, alpha=0.12, color='red')

ax.set_xlabel('Hour of Day (24 h)', fontsize=12)
ax.set_ylabel('Revenue per Mile (USD)', fontsize=12)
ax.set_title('NYC Taxi — Efficiency Index: Revenue per Mile by Hour (Jan 2024)',
             fontsize=13, fontweight='bold')
ax.set_xticks(range(0, 24))
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.2f'))

plt.tight_layout()
plt.savefig(os.path.join(KPI_DIR, 'kpi_efficiency_index.png'), bbox_inches='tight')
plt.show()

best_hour = kpi_efficiency.loc[kpi_efficiency['avg_rev_per_mile'].idxmax(), 'hour_of_day']
best_rpm  = kpi_efficiency['avg_rev_per_mile'].max()
print(f'Most efficient hour : {best_hour:02d}:00  (${best_rpm:.2f}/mile on average)')
print('\n── Efficiency Index (Top 5 Hours) ───────────────────────────────────')
print(kpi_efficiency.sort_values('avg_rev_per_mile', ascending=False).head(5).to_string(index=False))

### KPI 4 — Payment Trends

In [ ]:
kpi_payment = (
    df_gold
    .groupby('payment_label')
    .agg(
        trip_count=('payment_label', 'count'),
        total_revenue=('total_amount', 'sum'),
        avg_tip_pct=('tip_pct', 'mean'),
    )
    .reset_index()
)
kpi_payment['share_pct'] = (kpi_payment['trip_count'] / kpi_payment['trip_count'].sum() * 100).round(2)
kpi_payment = kpi_payment.sort_values('trip_count', ascending=False)

save_kpi(kpi_payment.set_index('payment_label'), 'kpi_payment_trends')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie — trip share
pie_colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#95a5a6']
wedges, texts, autotexts = axes[0].pie(
    kpi_payment['trip_count'],
    labels=kpi_payment['payment_label'],
    autopct='%1.1f%%',
    startangle=140,
    colors=pie_colors[:len(kpi_payment)],
    pctdistance=0.82,
    wedgeprops=dict(linewidth=1.5, edgecolor='white')
)
for at in autotexts:
    at.set_fontsize(9)
axes[0].set_title('Trip Share by Payment Method', fontsize=13, fontweight='bold')

# Bar — avg tip % by payment
top_payment = kpi_payment[kpi_payment['avg_tip_pct'] > 0]
bars = axes[1].bar(top_payment['payment_label'],
                   top_payment['avg_tip_pct'],
                   color=pie_colors[:len(top_payment)],
                   edgecolor='white')
axes[1].set_ylabel('Average Tip (%)', fontsize=11)
axes[1].set_title('Average Tip % by Payment Method', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Payment Method', fontsize=11)
for bar, val in zip(bars, top_payment['avg_tip_pct']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', fontsize=9)

plt.suptitle('NYC Taxi — Payment Trends (Jan 2024)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(KPI_DIR, 'kpi_payment_trends.png'), bbox_inches='tight')
plt.show()

print('\n── Payment Method Breakdown ─────────────────────────────────────────')
print(kpi_payment.to_string(index=False))

### Consolidated KPI Dashboard

In [ ]:
print('\n')
print('╔══════════════════════════════════════════════════════════════════════╗')
print('║              NYC TAXI ETL PIPELINE — FINAL KPI SUMMARY              ║')
print('╠══════════════════════════════════════════════════════════════════════╣')

peak_h   = kpi_busiest_hour.loc[kpi_busiest_hour['trip_count'].idxmax(), 'hour_of_day']
peak_c   = kpi_busiest_hour['trip_count'].max()
top_boro = kpi_borough_revenue.sort_values('total_revenue', ascending=False).iloc[0]
best_eff = kpi_efficiency.loc[kpi_efficiency['avg_rev_per_mile'].idxmax()]
top_pay  = kpi_payment.iloc[0]
cc_row   = kpi_payment[kpi_payment['payment_label'] == 'Credit Card']
cash_row = kpi_payment[kpi_payment['payment_label'] == 'Cash']

print(f'║  Gold dataset rows   : {len(df_gold):>10,}  (from {RAW_COUNT:,} raw)                  ║')
print(f'║  Rows removed        : {RAW_COUNT - len(df_gold):>10,}  ({(RAW_COUNT-len(df_gold))/RAW_COUNT*100:.2f}% filtered out)          ║')
print('╠══════════════════════════════════════════════════════════════════════╣')
print(f'║  Busiest hour        : {peak_h:02d}:00 – {peak_h+1:02d}:00   ({peak_c:,} trips)                 ║')
print(f'║  Top revenue borough : {top_boro["pickup_borough"]:<12}  (${top_boro["total_revenue"]/1e6:.2f}M total)          ║')
print(f'║  Best efficiency hr  : {int(best_eff["hour_of_day"]):02d}:00          (${best_eff["avg_rev_per_mile"]:.2f}/mile avg)          ║')
if not cc_row.empty:
    print(f'║  Credit card share   : {cc_row.iloc[0]["share_pct"]:.1f}%                                          ║')
if not cash_row.empty:
    print(f'║  Cash share          : {cash_row.iloc[0]["share_pct"]:.1f}%                                          ║')
print('╠══════════════════════════════════════════════════════════════════════╣')
print(f'║  Gold Parquet        : output/gold/nyc_taxi_gold.parquet            ║')
print(f'║  KPI CSVs + charts   : output/kpi/                                  ║')
print('╚══════════════════════════════════════════════════════════════════════╝')